# BirdCLEF+ 2025 — Jamie ablation training

Same model and training recipe as the baseline (EfficientNet-B0, AdamW 1e-3, one-cycle,
batch 64, 15 epochs, stratified 80/20 split with seed 42). **Exactly one thing changes per
experiment**, chosen by `EXPERIMENT` below, so any change in score comes from that switch.

| `EXPERIMENT` | Switch turned on | Problem it targets |
|---|---|---|
| `base` | none: checks the new data pipeline reproduces the baseline (val AUC ≈ 0.973) | — |
| `crop` | random 5 s crop from the whole clip instead of the first 5 s | weak labels |
| `noise` | add unlabelled-soundscape noise + mixup | domain shift, overfitting |
| `seclabel` | secondary labels as soft targets | label noise |
| `focal` | focal loss instead of BCE | class imbalance |
| `v2` | crop + noise/mixup + seclabel (each beat the baseline on the public LB) | all |
| `v2-nocrop` / `v2-nonoise` / `v2-nosec` | v2 with one switch removed (remove-one ablation) | — |
| `v2-focal` | v2 + focal loss (does focal help once overfitting is fixed?) | class imbalance |

**Validation is identical in every experiment** (first 5 s of each hold-out clip, primary
label only, macro ROC-AUC), so local scores can be compared fairly. Remember that local AUC
overstates the leaderboard; the public LB is the real test.

**Input:** the output of the `sc4000-birdclef-m1-make-mels` kernel (whole-clip mels).
**Output:** `model.pth`, `cfg.json`, `species.json` (what the inference notebook needs),
plus `history.csv` and a training-curve figure.

> Edit this master copy only (`notebooks/jamie_train/train.ipynb`), then run
> `python scripts/jamie_experiments.py` to refresh the per-experiment folders.

In [ ]:
EXPERIMENT = "seclabel"   # base | crop | noise | seclabel | focal | v2 | v2-nocrop | v2-nonoise | v2-nosec | v2-focal

In [ ]:
SWITCHES = {
    "base":     dict(),
    "crop":     dict(random_crop=True),
    "noise":    dict(noise=True, mixup=True),
    "seclabel": dict(secondary=True),
    "focal":    dict(focal=True),
    # v2 = the switches that beat the baseline on their own on the public LB
    # (crop +0.035, noise +0.016, seclabel +0.012; focal -0.012 was left out)
    "v2":         dict(random_crop=True, noise=True, mixup=True, secondary=True),
    # remove-one ablations of v2, plus a second chance for focal once overfitting is fixed
    "v2-nocrop":  dict(noise=True, mixup=True, secondary=True),
    "v2-nonoise": dict(random_crop=True, secondary=True),
    "v2-nosec":   dict(random_crop=True, noise=True, mixup=True),
    "v2-focal":   dict(random_crop=True, noise=True, mixup=True, secondary=True, focal=True),
}

CFG = dict(
    # mel settings (must match make_mels and the inference notebook)
    sr=32_000, duration=5, n_fft=2048, hop_length=512, n_mels=128, fmin=50, fmax=14_000,
    # training (same as the baseline)
    model="efficientnet_b0", epochs=15, batch_size=64, lr=1e-3, weight_decay=1e-2,
    val_frac=0.2, seed=42,
    # switches (all off = baseline)
    random_crop=False,
    noise=False, noise_prob=0.5, noise_db=(-20, -3),   # noise level relative to the clip, dB
    mixup=False, mixup_prob=0.5, mixup_alpha=0.4,
    secondary=False, secondary_weight=0.5,
    focal=False, focal_gamma=2.0,
)
CFG.update(SWITCHES[EXPERIMENT])
CFG["experiment"] = EXPERIMENT
print(EXPERIMENT, "->", {k: v for k, v in CFG.items()
                         if k in ("random_crop", "noise", "mixup", "secondary", "focal")})

In [ ]:
import os, ast, glob, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG["seed"])

def find_dir(marker, fallback):
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    return Path(hits[0]).parent if hits else Path(fallback)

DATA = Path(os.environ["M1_DATA_DIR"]) if os.environ.get("M1_DATA_DIR") else \
    find_dir("birdclef-2025/sample_submission.csv", "../../data")
MELS = Path(os.environ["M1_MELS_DIR"]) if os.environ.get("M1_MELS_DIR") else \
    find_dir("train_mels.npy", "../../outputs/jamie_make_mels")
OUT = Path("."); FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Data:", DATA, "| mels:", MELS, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## Load the pre-computed spectrograms
About 4.6 GB, loaded into RAM once (takes a minute or two instead of the baseline's 17).

In [ ]:
t0 = time.time()
X = np.load(MELS / "train_mels.npy")                 # (total_frames, 128) uint8
index = pd.read_csv(MELS / "train_index.csv")
mcfg = json.load(open(MELS / "mels_cfg.json"))
assert "store_db" in mcfg, "old make_mels output (80 dB); rerun make_mels version 2"
for k in ("sr", "n_fft", "hop_length", "n_mels", "fmin", "fmax"):
    assert mcfg[k] == CFG[k], f"mel setting {k} differs from make_mels"
NOISE = np.load(MELS / "noise_mels.npy") if CFG["noise"] else None
W = mcfg["frames_per_5s"]                            # 313 frames = 5 s
print(f"mels {X.shape} {X.nbytes / 1e9:.1f} GB | {len(index)} clips | "
      f"noise {None if NOISE is None else NOISE.shape} | {time.time() - t0:.0f}s")

## Labels
Species order follows `sample_submission.csv`, so outputs line up with submission columns.
With `secondary=True`, species in a clip's `secondary_labels` get a soft target of
`secondary_weight` (0.5 = "probably present") instead of 0.

In [ ]:
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]
print(len(index), "clips,", len(species), "species |",
      f"{np.mean([len(s) > 0 for s in sec_idx]):.1%} clips have usable secondary labels")

## Train / validation split (identical to the baseline)

In [ ]:
idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
print("train", len(idx_tr), "| val", len(idx_va))

## Dataset: cropping and background noise
* Each 5 s window is normalised **exactly like the inference notebook** does for every test
  window: 0 dB = the window's own loudest point, keep 80 dB below it, scale to [0, 1].
  (make_mels stores 120 dB, so quiet crops keep their faint background detail.)
* `random_crop`: a random 5 s window from anywhere in the (up to 30 s) clip; otherwise the
  first 5 s, like the baseline.
* `noise`: with probability `noise_prob`, a random 5 s piece of an unlabelled soundscape is
  added. The mixing is done on the power scale (the real "adding two sounds"), at a random
  level between -20 and -3 dB below the clip.

In [ ]:
TOP_DB = 80.0                       # model input keeps 80 dB, like the baseline
STORE_DB = float(mcfg["store_db"])  # make_mels stored 120 dB below each clip's peak

def to_db(u8):
    """uint8 from make_mels -> dB relative to the clip's loudest point."""
    return u8.T.astype(np.float32) / 255.0 * STORE_DB - STORE_DB

def to_input(db):
    """Same as inference: 0 dB = this window's loudest point, keep 80 dB below it, scale to [0, 1]."""
    return ((np.maximum(db - db.max(), -TOP_DB) + TOP_DB) / TOP_DB).astype(np.float32)

START, NFRAMES = index["start"].values, index["n_frames"].values

class MelDataset(torch.utils.data.Dataset):
    def __init__(self, idx, train):
        self.idx, self.train = idx, train
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        s, n = START[j], NFRAMES[j]
        off = np.random.randint(0, n - W + 1) if (self.train and CFG["random_crop"]) else 0
        db = to_db(X[s + off : s + off + W])                 # (128, W)

        if self.train and CFG["noise"] and np.random.rand() < CFG["noise_prob"]:
            k = np.random.randint(len(NOISE))
            o = np.random.randint(0, NOISE.shape[1] - W + 1)
            nz = to_db(NOISE[k, o : o + W])
            gain_db = np.random.uniform(*CFG["noise_db"])
            # add the two sounds on the power scale, noise peak gain_db below the clip peak
            p = 10.0 ** ((db - db.max()) / 10.0) + 10.0 ** ((nz - nz.max() + gain_db) / 10.0)
            db = 10.0 * np.log10(p)
        x = to_input(db)

        target = np.zeros(len(species), dtype=np.float32)
        if self.train and CFG["secondary"]:
            target[sec_idx[j]] = CFG["secondary_weight"]
        target[y_idx[j]] = 1.0          # validation always uses the primary label only
        return torch.from_numpy(x)[None], torch.from_numpy(target)

NW = int(os.environ.get("M1_WORKERS", min(4, os.cpu_count())))   # 0 for a local Mac test
dl_tr = torch.utils.data.DataLoader(MelDataset(idx_tr, True), batch_size=CFG["batch_size"],
                                    shuffle=True, num_workers=NW, drop_last=True,
                                    persistent_workers=NW > 0)
dl_va = torch.utils.data.DataLoader(MelDataset(idx_va, False), batch_size=CFG["batch_size"] * 2,
                                    shuffle=False, num_workers=NW)

## Mixup and focal loss
* `mixup`: with probability `mixup_prob` per batch, each clip is blended with another clip
  from the same batch (weight drawn from Beta(0.4, 0.4)). Both species count as present, so
  the target is the element-wise maximum of the two targets.
* `focal`: focal loss multiplies each BCE term by (1 - p_t)^gamma, so examples the model
  already gets right count less and training focuses on hard (often rare) ones.
  Validation loss is always plain BCE, so it can be compared with the baseline.

In [ ]:
def mixup(xb, yb):
    lam = float(np.random.beta(CFG["mixup_alpha"], CFG["mixup_alpha"]))
    perm = torch.randperm(xb.size(0), device=xb.device)
    return lam * xb + (1 - lam) * xb[perm], torch.maximum(yb, yb[perm])

class FocalBCE(nn.Module):
    def __init__(self, gamma):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * target + (1 - p) * (1 - target)
        return ((1 - p_t) ** self.gamma * bce).mean()

bce = nn.BCEWithLogitsLoss()
train_criterion = FocalBCE(CFG["focal_gamma"]) if CFG["focal"] else bce

## Model and training loop (same as the baseline)

In [ ]:
model = timm.create_model(CFG["model"], pretrained=not os.environ.get("M1_NO_PRETRAINED"),
                          in_chans=1, num_classes=len(species)).to(DEVICE)
EPOCHS = int(os.environ.get("M1_EPOCHS", CFG["epochs"]))
MAX_STEPS = int(os.environ.get("M1_MAX_STEPS", "0"))    # >0 only for a quick local test
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG["lr"], total_steps=EPOCHS * len(dl_tr), pct_start=0.1)
scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro")

@torch.no_grad()
def evaluate():
    model.eval()
    preds, targets, loss_sum = [], [], 0.0
    for step, (xb, yb) in enumerate(dl_va):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            logits = model(xb)
        loss_sum += bce(logits.float(), yb).item() * len(xb)
        preds.append(torch.sigmoid(logits.float()).cpu().numpy()); targets.append(yb.cpu().numpy())
        if MAX_STEPS and step + 1 >= MAX_STEPS:
            break
    p, t = np.concatenate(preds), np.concatenate(targets)
    return loss_sum / len(p), macro_auc(t, p)

history, best_auc = [], 0.0
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); loss_sum, n_seen = 0.0, 0
    for step, (xb, yb) in enumerate(dl_tr):
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        if CFG["mixup"] and np.random.rand() < CFG["mixup_prob"]:
            xb, yb = mixup(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
            loss = train_criterion(model(xb).float(), yb)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        loss_sum += loss.item() * len(xb); n_seen += len(xb)
        if MAX_STEPS and step + 1 >= MAX_STEPS:
            break
    tr_loss = loss_sum / n_seen
    va_loss, va_auc = evaluate()
    history.append(dict(epoch=epoch, train_loss=tr_loss, val_loss=va_loss, val_auc=va_auc))
    flag = ""
    if va_auc > best_auc:
        best_auc = va_auc; torch.save(model.state_dict(), OUT / "model.pth"); flag = "  <- saved"
    print(f"epoch {epoch:2d} | train {tr_loss:.4f} | val {va_loss:.4f} | val AUC {va_auc:.4f} | "
          f"{time.time() - t0:.0f}s{flag}", flush=True)

print(f"[{EXPERIMENT}] best val macro ROC-AUC: {best_auc:.4f} "
      f"(epoch {max(history, key=lambda h: h['val_auc'])['epoch']})")

## Save config and training curves

In [ ]:
cfg_out = dict(CFG, best_val_auc=best_auc)
cfg_out["noise_db"] = list(CFG["noise_db"])
json.dump(cfg_out, open(OUT / "cfg.json", "w"), indent=1)
json.dump(species, open(OUT / "species.json", "w"))
hist = pd.DataFrame(history); hist.to_csv(OUT / "history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
hist.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0],
          title="Loss (train: " + ("focal" if CFG["focal"] else "BCE") + ", val: BCE)")
hist.plot(x="epoch", y="val_auc", ax=axes[1], title=f"Validation macro ROC-AUC ({EXPERIMENT})",
          legend=False)
plt.tight_layout(); plt.savefig(FIG / f"training_{EXPERIMENT}.png", dpi=200); plt.show()